# TFM — Estudio Comparativo del Espacio Latente de Modelos Generativos de Texto-Imagen

## Notebook de configuración del entorno en Google Colab

**Instrucciones antes de ejecutar:**
Ir a `Entorno de ejecución > Cambiar tipo de entorno de ejecución` y seleccionar **GPU T4**.


**Estructura de este notebook:**
- 1 — Verificación de GPU
- 2 — Montaje de Google Drive y rutas del proyecto
- 3 — Instalación de dependencias
- 4 — Verificación de importaciones


---
## 1 — Verificación de GPU

In [ ]:
# Verificación inmediata de disponibilidad de GPU antes de instalar dependencias.
# Sin GPU T4 (15 GB VRAM), la carga de los modelos de difusión no es viable.
# El script aborta aquí con un mensaje claro en lugar de fallar silenciosamente
# al intentar cargar los pesos.
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU no disponible. "
        "Ve a Entorno de ejecución > Cambiar tipo de entorno de ejecución "
        "y selecciona GPU T4."
    )

gpu_name  = torch.cuda.get_device_name(0)
vram_gb   = torch.cuda.get_device_properties(0).total_memory / 1024**3
torch_ver = torch.__version__
cuda_ver  = torch.version.cuda

print(f"GPU     : {gpu_name}")
print(f"VRAM    : {vram_gb:.1f} GB")
print(f"PyTorch : {torch_ver}")
print(f"CUDA    : {cuda_ver}")

---
## 2 — Montaje de Google Drive y rutas del proyecto

El proyecto vive en `Mi unidad/TFM/` dentro de Google Drive.  
Todos los datos, resultados y checkpoints se guardan allí para persistir entre sesiones de Colab.

In [ ]:
# Montaje de Google Drive y creación de la estructura de directorios del proyecto.
# Todos los datos persistentes (latentes, resultados, checkpoints LoRA, figuras)
# se almacenan en Drive para sobrevivir entre sesiones. El código fuente también
# se sincroniza aquí (ver §5). Solo los pesos de los modelos (~16 GB totales)
# se cachean en disco local de Colab para no consumir cuota de Drive.
from google.colab import drive
import os

drive.mount("/content/drive", force_remount=False)

# Raíz del proyecto en Drive
PROJECT_ROOT = "/content/drive/MyDrive/TFM/repositorio"

# Subdirectorios del proyecto
DIRS = [
    "models",
    "finetuning",
    "metrics",
    "visualizations",
    "ajuste_fino/dataset_wikiart",
    "data/generated",
    "corpus/latents",
    "notebooks",
    "memoria",
]

for d in DIRS:
    os.makedirs(os.path.join(PROJECT_ROOT, d), exist_ok=True)

print(f"Proyecto en: {PROJECT_ROOT}")
print("Estructura de directorios lista.")

In [ ]:
import sys, os

# Raíz del proyecto en Drive
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Los pesos de los modelos se cachean en el disco LOCAL de Colab (/content/),
# no en Drive. Esto evita consumir los ~15 GB del plan gratuito con los pesos
# (~16 GB en total para los 3 modelos). El coste es re-descargar al inicio de
# cada sesión (~10-15 min); HuggingFace → Colab es muy rápido.
#
# Solo van a Drive: código, latentes, resultados, figuras y checkpoints LoRA.
os.environ["HF_HOME"] = "/content/.cache/huggingface"

print("HF_HOME (modelos, efímero) :", os.environ["HF_HOME"])
print("PROJECT_ROOT   (Drive)     :", PROJECT_ROOT)

---
## 3 — Instalación de dependencias

PyTorch viene preinstalado en Colab. Solo se instalan las bibliotecas adicionales del proyecto.  
En sesiones posteriores se puede saltar si los paquetes ya están instalados (Colab los pierde al cerrar la sesión).

In [ ]:
# Instalación de dependencias desde requirements_colab.txt.
# preinstaladas en Colab y las requeridas por el proyecto (p. ej. diffusers
# y transformers se actualizan frecuentemente y sus APIs cambian entre versiones).
# Colab pierde los paquetes al cerrar la sesión.
import os
req_file = os.path.join(PROJECT_ROOT, "requirements_colab.txt")

if os.path.exists(req_file):
    print("Instalando/actualizando desde requirements_colab.txt...")
    # -U fuerza la actualización de paquetes ya instalados en Colab,
    # resolviendo conflictos entre versiones del sistema y las nuestras.
    !pip install -q -U -r {req_file}
else:
    print("requirements_colab.txt no encontrado. Instalando versiones mínimas...")
    !pip install -q -U \
        "diffusers>=0.30.0" \
        "transformers>=4.47.0" \
        "accelerate>=1.0.0" \
        "safetensors>=0.4.3" \
        "peft>=0.12.0" \
        "bitsandbytes>=0.43.1" \
        "lpips==0.1.4" \
        "open-clip-torch>=2.24.0" \
        "huggingface-hub>=0.23.2" \
        "datasets>=2.19.1" \
        einops

print("Instalación completada.")

---
## 4 — Verificación de importaciones

Comprueba que todos los paquetes se importan correctamente y que la GPU está operativa.

In [ ]:
import importlib, os, torch

os.environ.setdefault("TORCHDYNAMO_DISABLE", "1")

# --- Paquetes obligatorios (deben importar para continuar) ---
required = [
    ("torch",        "PyTorch"),
    ("diffusers",    "Diffusers"),
    ("transformers", "Transformers"),
    ("peft",         "PEFT"),
    ("bitsandbytes", "bitsandbytes"),
    ("lpips",        "LPIPS"),
    ("open_clip",    "OpenCLIP"),
    ("datasets",     "Datasets"),
    ("sklearn",      "scikit-learn"),
    ("numpy",        "NumPy"),
    ("matplotlib",   "matplotlib"),
    ("PIL",          "Pillow"),
    ("einops",       "einops"),
]

# --- Paquetes opcionales (WARN si fallan, no bloquean) ---
# accelerate: necesario para fine-tuning (Fase 6), no para inferencia
# xformers:   versión debe coincidir exactamente con PyTorch
optional = [
    ("accelerate", "Accelerate",
     "solo necesario en Fase 6 (fine-tuning); bug conocido en PyTorch 2.11.x"),
    ("xformers",   "xformers",
     "no instalado — se usará atención estándar (impacto mínimo con T4)"),
]

all_ok = True
for module, name in required:
    try:
        mod = importlib.import_module(module)
        ver = getattr(mod, "__version__", "n/a")
        print(f"  OK   {name:<20} {ver}")
    except ImportError as e:
        print(f"  FAIL {name:<20} {e}")
        all_ok = False

for module, name, note in optional:
    try:
        mod = importlib.import_module(module)
        ver = getattr(mod, "__version__", "n/a")
        print(f"  OK   {name:<20} {ver}  (opcional)")
    except (ImportError, AttributeError):
        print(f"  WARN {name:<20} no disponible — {note}")

print()
print(f"GPU disponible : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU activa     : {props.name}")
    print(f"VRAM total     : {props.total_memory / 1024**3:.1f} GB")
print()
print("Entorno listo para inferencia." if all_ok else "Revisa los paquetes marcados como FAIL.")

In [ ]:
# Prueba de carga mínima: tensor en GPU
import torch

x = torch.randn(4, 4, device="cuda")
y = torch.randn(4, 4, device="cuda")
z = x @ y
print("Multiplicación matricial en GPU correcta. Resultado shape:", z.shape)
print(f"Memoria GPU usada: {torch.cuda.memory_allocated() / 1024**2:.1f} MB")

---
## 5 — Resumen del entorno

Celda de resumen para registrar el estado del entorno al inicio de cada sesión.

In [ ]:
import platform, datetime
import torch, diffusers, transformers, peft

print("=" * 50)
print(" RESUMEN DEL ENTORNO TFM")
print("=" * 50)
print(f" Fecha         : {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f" Python        : {platform.python_version()}")
print(f" PyTorch       : {torch.__version__}")
print(f" CUDA          : {torch.version.cuda}")
print(f" Diffusers     : {diffusers.__version__}")
print(f" Transformers  : {transformers.__version__}")
print(f" PEFT          : {peft.__version__}")
print("-" * 50)
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f" GPU           : {props.name}")
    print(f" VRAM total    : {props.total_memory / 1024**3:.1f} GB")
    print(f" VRAM libre    : {(props.total_memory - torch.cuda.memory_allocated()) / 1024**3:.1f} GB")
print("-" * 50)
print(f" PROJECT_ROOT  : {PROJECT_ROOT}")
print("=" * 50)